In [1]:
import sys
sys.path.insert(0,'C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hp\\20261019T153420-0d8a8f19')
import kernel_identity as _kernel_management
_kernel_management.write_kernel_runtime_identity('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hp\\20261019T153420-0d8a8f19',40752)
import local_course_adapter as _local_adapter
_local_adapter.install('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hp\\20261019T153420-0d8a8f19\\source\\HaiLong_Colab_Exercises','C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hp\\20261019T153420-0d8a8f19')


# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original fast.ai competition exercise. Upload paddy-disease-classification.zip from Kaggle when prompted. No API credentials are written and nothing is submitted. Batch sizes are reduced for Colab GPU memory; full experiments can take hours.

Source: https://www.kaggle.com/code/jhoward/multi-target-road-to-the-top-part-4

In [2]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '13_paddy_multi_target.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: C:\Users\[LOCAL_USER]\.v\hp\20261019T153420-0d8a8f19\source\HaiLong_Colab_Exercises\03_fastai
Run mode: FULL EXERCISE
Results: C:\Users\[LOCAL_USER]\.v\hp\20261019T153420-0d8a8f19\source\HaiLong_Colab_Exercises\03_fastai\results


## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Supply the original competition ZIP after accepting its rules; predictions and submission CSVs are local evidence, not submission receipts.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


In [3]:
from colab_runtime import competition_data
iskaggle = False

This is part 4 of the [Road to the Top](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1) series, in which I show the process I used to tackle the [Paddy Doctor](https://www.kaggle.com/competitions/paddy-disease-classification) competition. The first three parts show the process I took that lead to four 1st place submissions. This part onwards discusses various extensions and advanced tricks you might want to consider -- but which I don't promise will necessarily help your score (you'll need to do your own experiments to figure this out!) If you haven't read the rest of this series yet, I'd suggest starting at [part 1](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1).

In this notebook we're going to build a model that doesn't just predict what disease the rice paddy has, but also predicts what kind of rice is shown.

This might sound like a bad idea. After all, doesn't that mean that the model has *more* to do? Mightn't it get rather distracted from its main task, which is to identify paddy disease?

Perhaps... But in previous projects I've often found the opposite to be true, especially when training for quite a few epochs. By giving the model more signal about what is present in a picture, it may be able to use this information to find more interesting features that predict our target of interest. For instance, perhaps some of the features of disease change between varieties.

## Multi-output `DataLoader`

First we'll repeat the steps we used last time to access the data and ensure all the latest libraries are installed:

In [4]:
comp = 'paddy-disease-classification'
path = competition_data(comp)
from fastai.vision.all import *
set_seed(42)

from fastcore.parallel import *
trn_path = path/'train_images'

Here's the CSV that Kaggle provides, showing the variety of rice contained in each image -- we'll make `image_id` the index of our data frame so that we can look up images directly to grab their variety:

In [5]:
df = pd.read_csv(path/'train.csv', index_col='image_id')
df.head()

,label,variety,age
image_id,,,
100330.jpg,bacterial_leaf_blight,ADT45,45
100365.jpg,bacterial_leaf_blight,ADT45,45
100382.jpg,bacterial_leaf_blight,ADT45,45
100632.jpg,bacterial_leaf_blight,ADT45,45
101918.jpg,bacterial_leaf_blight,ADT45,45


Pandas uses the `loc` attribute to look up rows by index. Here's how we can get the variety of image `100330.jpg`, for instance:

In [6]:
df.loc['100330.jpg', 'variety']

'ADT45'

Our DataBlock will be using `get_image_files` to get the list of training images, which returns `Path` objects. Therefore, to look up an item to get its variety, we'll need to pass its `name`. Here's a function which does just that:

In [7]:
def get_variety(p): return df.loc[p.name, 'variety']

We're now ready to create our `DataLoaders`. To do this, we'll use the `DataBlock` API, which is a flexible and convenient way to plug pieces of a data processing pipeline together:

In [8]:
dls = DataBlock(
    blocks=(ImageBlock,CategoryBlock,CategoryBlock),
    n_inp=1,
    get_items=get_image_files,
    get_y = [parent_label,get_variety],
    splitter=RandomSplitter(0.2, seed=42),
    item_tfms=Resize(192, method='squish'),
    batch_tfms=aug_transforms(size=128, min_scale=0.75)
).dataloaders(trn_path,bs=8)

Here's an explanation of each line:

```python
blocks=(ImageBlock,CategoryBlock,CategoryBlock),
```

The `DataBlock` will create 3 things from each file: an image (the contents of the file), and 2 categorical variables (the disease and the variety).

```python
n_inp=1,
```

There is `1` input (the image) -- and therefore the other two variables (the two categories) are outputs.

```python
get_items=get_image_files,
```

Use `get_image_files` to get a list of inputs.

```python
get_y = [parent_label,get_variety],
```

To create the two outputs for each file, call two functions: `parent_label` (from fastai) and `get_variety` (defined above).

```python
splitter=RandomSplitter(0.2, seed=42),
```

Randomly split the input into 80% train and 20% validation sets.

```python
item_tfms=Resize(192, method='squish'),
batch_tfms=aug_transforms(size=128, min_scale=0.75)
```

These are the same item and batch transforms we've used in previous notebooks.

Let's take a look at part of a batch of this data:

In [9]:
dls.show_batch(max_n=6)

We can see that fastai has created both the image input and two categorical outputs that we requested!

## Replicating the disease model

Now we'll replicate the same disease model we've made before, but have it work with this new data.

The key difference is that our metrics and loss will now receive three things instead of two: the model outputs (i.e. the metric and loss function inputs), and the two targets (disease and variety). Therefore, we need to define slight variations of our metric (`error_rate`) and loss function (`cross_entropy`) to pass on just the `disease` target:

In [10]:
def disease_err(inp,disease,variety): return error_rate(inp,disease)
def disease_loss(inp,disease,variety): return F.cross_entropy(inp,disease)

We're now ready to create our learner.

There's just one wrinkle to be aware of. Now that our `DataLoaders` is returning multiple targets, fastai doesn't know how many outputs our model will need. Therefore we have to pass `n_out` when we create our `Learner` -- we need `10` outputs, one for each possible disease:

In [11]:
arch = 'convnext_small.fb_in22k'
learn = vision_learner(dls, arch, loss_func=disease_loss, metrics=disease_err, n_out=10, cbs=GradientAccumulation(64)).to_fp16()
lr = 0.01

C:\Users\[LOCAL_USER]\.v\hlpaddy\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hp\20261019T153420-0d8a8f19\hf-cache\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  265MB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Users\[LOCAL_USER]\.v\hlpaddy\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hp\20261019T153420-0d8a8f19\hf-cache\hub\models--timm--convnext_small.fb_in22k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


When we train this model we should get similar results to what we've seen with similar models before:

In [12]:
learn.fine_tune(5, lr)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.666 GB; runtime free 557.068 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 557.068 GB. Budget is not Drive quota.


epoch,train_loss,valid_loss,disease_err,time
0,1.377727,0.819368,0.264296,02:59


Storage check (checkpoint_write): artifacts 0.202 GB; configured budget 80.000 GB; additional estimate 0.219 GB; runtime free 556.736 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.413 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 556.524 GB. Budget is not Drive quota.


Storage check (before_fit): artifacts 0.202 GB; configured budget 80.000 GB; additional estimate 2.446 GB; runtime free 556.735 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.202 GB; configured budget 80.000 GB; additional estimate 0.213 GB; runtime free 556.735 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 1.211 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 555.324 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.211 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 554.814 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.211 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 554.522 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.211 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 554.429 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.211 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 554.429 GB. Budget is not Drive quota.


## Multi-target model

In order to predict both the probability of each disease, and of each variety, we'll now need the model to output a tensor of length 20, since there are 10 possible diseases, and 10 possible varieties. We can do this by setting `n_out=20`:

In [13]:
learn = vision_learner(dls, arch, n_out=20).to_fp16()

We can define `disease_loss` just like we did previously, but with one important change: the input tensor is now length 20, not 10, so it doesn't match the number of possible diseases. We can pick whatever part of the input we want to be used to predict disease. Let's use the first 10 values:

In [14]:
def disease_loss(inp,disease,variety): return F.cross_entropy(inp[:,:10],disease)

That means we can do the same thing for predicting variety, but use the last 10 values of the input, and set the target to `variety` instead of `disease`:

In [15]:
def variety_loss(inp,disease,variety): return F.cross_entropy(inp[:,10:],variety)

Our overall loss will then be the sum of these two losses:

In [16]:
def combine_loss(inp,disease,variety): return disease_loss(inp,disease,variety)+variety_loss(inp,disease,variety)

It would be useful to view the error rate for each of the outputs too, so let's do the same thing for out metrics:

In [17]:
def disease_err(inp,disease,variety): return error_rate(inp[:,:10],disease)
def variety_err(inp,disease,variety): return error_rate(inp[:,10:],variety)

err_metrics = (disease_err,variety_err)

It's useful to see the loss for each of the outputs too, so we'll add those as metrics:

In [18]:
all_metrics = err_metrics+(disease_loss,variety_loss)

We're now ready to create and train our `Learner`:

In [19]:
learn = vision_learner(dls, arch, loss_func=combine_loss, metrics=all_metrics, n_out=20, cbs=GradientAccumulation(64)).to_fp16()

In [20]:
learn.fine_tune(5, lr)

Storage check (before_fit): artifacts 0.403 GB; configured budget 80.000 GB; additional estimate 0.667 GB; runtime free 555.237 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.403 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 555.237 GB. Budget is not Drive quota.


epoch,train_loss,valid_loss,disease_err,variety_err,disease_loss,variety_loss,time
0,2.524914,1.501923,0.325805,0.182605,0.946948,0.554976,01:50


Storage check (checkpoint_write): artifacts 0.605 GB; configured budget 80.000 GB; additional estimate 0.219 GB; runtime free 555.035 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.816 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 554.824 GB. Budget is not Drive quota.

Storage check (before_fit): artifacts 0.605 GB; configured budget 80.000 GB; additional estimate 2.446 GB; runtime free 555.035 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.605 GB; configured budget 80.000 GB; additional estimate 0.213 GB; runtime free 555.035 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 1.615 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 552.775 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.615 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 549.986 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.615 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 531.071 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.615 GB; configured budget 80.000 GB; additional estimate 0.812 GB; runtime free 528.552 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 1.615 GB; configured budget 80.000 GB; additional estimate 0.209 GB; runtime free 528.552 GB. Budget is not Drive quota.


## Conclusion

So, is this useful?

Well... if you actually want a model that predicts multiple things, then yes, definitely! But as to whether it's going to help us better predict rice disease, I honestly don't know. I haven't come across any research that tackles this important question: when can a multi-target model improve the accuracy of the individual targets compared to a single target model? (That doesn't mean it doesn't exist of course -- perhaps it does and I haven't found it yet...)

I've certainly found in previous projects that there are cases where improvements to single targets can be made by using a multi-target model. I'd guess that it'll be most useful when you're having problems with overfitting. So try doing this with more epochs, and let me know how you go!

If you found this notebook useful, please remember to click the little up-arrow at the top to upvote it, since I like to know when people have found my work useful, and it helps others find it too. And if you have any questions or comments, please pop them below -- I read every comment I receive.

In [21]:
print("Predictions stay in this notebook. No competition submission or publication is performed.")

Predictions stay in this notebook. No competition submission or publication is performed.


Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [22]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [23]:
_local_adapter.probe()


__HAILONG_LOCAL_PART1_EVIDENCE__{"backend": "local_native_windows_cuda", "in_colab": false, "drive_enabled": false, "quick_run": false, "notebook": "03_fastai/13_paddy_multi_target.ipynb", "source_sha256": "7712f6387812f2ffa86b108a0c52067b4b0b19bfc555540b277b6a93ee63b9aa", "dataset_sha256": "defc373e450dff931459534e5e10e532a3c61336ab57c02972d6640d33156943", "source_rows": 10407, "fit_epochs": [1, 5, 1, 5], "fit_metadata": [{"fit_index": 0, "epochs": 1, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}, {"fit_index": 1, "epochs": 5, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}, {"fit_index": 2, "epochs": 1, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}, {"fit_index": 3, "epochs": 5, "train_rows": 8326, "validation_rows": 2081, "microbatch_size": 8}], "evaluation": {"disease_error_rate": 0.06439211964607239, "variety_error_rate": 0.016338298097252846, "heldout_rows": 2081, "final_weights_match_current_model": true}, "all_origina

{'backend': 'local_native_windows_cuda',
 'in_colab': False,
 'drive_enabled': False,
 'quick_run': False,
 'notebook': '03_fastai/13_paddy_multi_target.ipynb',
 'source_sha256': '7712f6387812f2ffa86b108a0c52067b4b0b19bfc555540b277b6a93ee63b9aa',
 'dataset_sha256': 'defc373e450dff931459534e5e10e532a3c61336ab57c02972d6640d33156943',
 'source_rows': 10407,
 'fit_epochs': [1, 5, 1, 5],
 'fit_metadata': [{'fit_index': 0,
   'epochs': 1,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8},
  {'fit_index': 1,
   'epochs': 5,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8},
  {'fit_index': 2,
   'epochs': 1,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8},
  {'fit_index': 3,
   'epochs': 5,
   'train_rows': 8326,
   'validation_rows': 2081,
   'microbatch_size': 8}],
 'evaluation': {'disease_error_rate': 0.06439211964607239,
  'variety_error_rate': 0.016338298097252846,
  'heldout_rows': 2081,
  'final_weights_mat